In [1]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Buy_Presure_Scaner import *
from Trade_Execution import *
from Chandelier_ZLSMA import *
from Chandelier_ZLSMA_Filter import *
from EMAs_9_15_Filter import *
from Fetch_Coin_List import *
from Consolidation_Scaner import *

In [2]:
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

def fetch_binance_data(symbol, timeframe='15m', limit=500):
    """
    Fetch historical candlestick data from Binance for a given symbol.
    
    Parameters:
    symbol: str - Trading pair (e.g., 'BTCUSDT')
    timeframe: str - Candlestick timeframe (default '15m')
    limit: int - Number of candles to fetch (default 500)
    
    Returns:
    pandas DataFrame - OHLCV data with timestamp and close price
    """
    try:
        exchange = ccxt.binance()
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
        
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        
        return df
    except Exception:
        return None

def calculate_ema(data, period):
    """Calculate Exponential Moving Average"""
    return data.ewm(span=period, adjust=False).mean()

def calculate_ema_cross(in_df, ema9_period=9, ema15_period=15, close_column='close', timestamp_column=None):
    """
    Calculate EMA9 and EMA15 with cross detection
    
    Parameters:
    in_df: pandas DataFrame - input DataFrame
    ema9_period: int - EMA9 period (default 9)
    ema15_period: int - EMA15 period (default 15)
    close_column: str - name of the close price column (default 'close')
    timestamp_column: str - name of the timestamp column (if None, will try to auto-detect)
    
    Returns:
    pandas DataFrame - DataFrame with original data and EMA columns
    """
    df = in_df.copy()
    
    # Handle timestamp column
    if timestamp_column is None:
        timestamp_candidates = ['timestamp', 'time', 'date', 'datetime', 'Date', 'Time', 'DateTime']
        for col in timestamp_candidates:
            if col in df.columns:
                timestamp_column = col
                break
    
    if timestamp_column and timestamp_column in df.columns:
        df[timestamp_column] = pd.to_datetime(df[timestamp_column])
    
    # Calculate EMAs
    df['ema9'] = calculate_ema(df[close_column], ema9_period)
    df['ema15'] = calculate_ema(df[close_column], ema15_period)
    
    # Calculate EMA cross signals
    df['ema9_above_ema15'] = df['ema9'] > df['ema15']
    df['ema9_below_ema15'] = df['ema9'] < df['ema15']
    
    # Detect crossovers - ENHANCED LOGIC
    # Golden cross: EMA9 crosses above EMA15
    df['golden_cross'] = (df['ema9'] > df['ema15']) & (df['ema9'].shift(1) <= df['ema15'].shift(1))
    # Death cross: EMA9 crosses below EMA15
    df['death_cross'] = (df['ema9'] < df['ema15']) & (df['ema9'].shift(1) >= df['ema15'].shift(1))
    
    # Price above/below EMAs
    df['price_above_ema9'] = df[close_column] > df['ema9']
    df['price_above_ema15'] = df[close_column] > df['ema15']
    df['price_above_both_emas'] = df['price_above_ema9'] & df['price_above_ema15']
    
    # EMA signal (1 for bullish, -1 for bearish, 0 for neutral)
    df['ema_signal'] = 0
    df.loc[df['ema9_above_ema15'] & df['price_above_both_emas'], 'ema_signal'] = 1
    df.loc[df['ema9_below_ema15'] & (~df['price_above_both_emas']), 'ema_signal'] = -1
    
    # Add EMA difference for strength analysis
    df['ema_diff'] = df['ema9'] - df['ema15']
    df['ema_diff_pct'] = (df['ema_diff'] / df['ema15']) * 100
    
    return df

class BinanceEMAAnalyzer:
    def __init__(self, api_key=None, api_secret=None):
        """Initialize Binance connection"""
        self.exchange = ccxt.binance({
            'apiKey': api_key,
            'secret': api_secret,
            'sandbox': False,
            'rateLimit': 1200,
            'enableRateLimit': True,
        })
        
    def normalize_symbol(self, symbol):
        """Convert symbol to proper format (e.g., 'BTC' -> 'BTCUSDT', 'BTCUSDT' -> 'BTCUSDT')"""
        symbol = symbol.upper().strip()
        
        # If it doesn't end with USDT, add it
        if not symbol.endswith('USDT'):
            symbol = symbol + 'USDT'
            
        return symbol
    
    def get_klines(self, symbol, timeframe='15m', limit=200):
        """Fetch OHLCV data for a symbol"""
        try:
            # Convert to ccxt format for API call
            ccxt_symbol = symbol[:-4] + '/' + symbol[-4:]  # BTCUSDT -> BTC/USDT
            
            ohlcv = self.exchange.fetch_ohlcv(ccxt_symbol, timeframe, limit=limit)
            if not ohlcv or len(ohlcv) < 50:
                return None
                
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            return df
        except Exception as e:
            print(f"Error fetching data for {symbol}: {str(e)}")
            return None
    
    def check_ema_cross_within_candles(self, df, max_candles=3):
        """
        Enhanced function to check if golden cross occurred within specified candles
        AND current price is above both EMAs
        
        Returns: (is_valid_signal, candles_since_cross, cross_details)
        """
        if df is None or len(df) < 20:  # Need at least 20 candles for reliable EMA
            return False, 0, None
        
        # Find all golden cross points
        golden_cross_indices = df[df['golden_cross'] == True].index.tolist()
        
        if not golden_cross_indices:
            return False, 0, None
        
        # Get the most recent golden cross
        latest_cross_idx = golden_cross_indices[-1]
        current_idx = df.index[-1]
        candles_since_cross = current_idx - latest_cross_idx
        
        # Check if cross is within the specified window
        if candles_since_cross > max_candles:
            return False, candles_since_cross, None
        
        # Verify current conditions
        current_row = df.iloc[-1]
        
        # All conditions must be met:
        # 1. EMA9 is above EMA15 (maintained after cross)
        # 2. Current price is above both EMAs
        # 3. Cross happened within max_candles
        if (current_row['ema9_above_ema15'] and 
            current_row['price_above_both_emas']):
            
            cross_details = {
                'cross_index': latest_cross_idx,
                'cross_time': df.iloc[latest_cross_idx]['timestamp'],
                'cross_price': df.iloc[latest_cross_idx]['close'],
                'ema9_at_cross': df.iloc[latest_cross_idx]['ema9'],
                'ema15_at_cross': df.iloc[latest_cross_idx]['ema15'],
                'price_change_since_cross': ((current_row['close'] - df.iloc[latest_cross_idx]['close']) / 
                                            df.iloc[latest_cross_idx]['close'] * 100)
            }
            
            return True, candles_since_cross, cross_details
        
        return False, candles_since_cross, None
    
    def analyze_symbols(self, symbols, timeframe='15m', limit=200, max_cross_candles=3, max_price=None):
        """
        Analyze specific symbols for EMA golden cross signals
        
        Parameters:
        symbols: list - List of symbols to analyze
        timeframe: str - Candlestick timeframe
        limit: int - Number of candles to fetch
        max_cross_candles: int - Maximum candles since golden cross
        max_price: float - Maximum price filter
        
        Returns:
        list - Analysis results for qualifying symbols
        """
        if not symbols:
            return []
            
        results = []
        
        print(f"Analyzing {len(symbols)} symbols...")
        
        for i, symbol in enumerate(symbols):
            try:
                # Normalize symbol format
                normalized_symbol = self.normalize_symbol(symbol)
                
                # Get price data
                df = self.get_klines(normalized_symbol, timeframe, limit)
                if df is None:
                    continue
                
                # Get current price
                current_price = df['close'].iloc[-1]
                
                # Apply price filter if specified
                if max_price is not None and current_price >= max_price:
                    continue
                
                # Calculate EMA indicators
                df = calculate_ema_cross(df)
                
                # Check for valid EMA cross signal
                is_valid, candles_since, cross_details = self.check_ema_cross_within_candles(
                    df, max_cross_candles
                )
                
                current_row = df.iloc[-1]
                
                # Only add to results if it meets ALL criteria
                if is_valid:
                    result = {
                        'symbol': normalized_symbol,
                        'current_price': current_price,
                        'ema9': current_row['ema9'],
                        'ema15': current_row['ema15'],
                        'candles_since_cross': candles_since,
                        'has_valid_signal': is_valid,
                        'cross_time': cross_details['cross_time'] if cross_details else None,
                        'cross_price': cross_details['cross_price'] if cross_details else None,
                        'price_change_pct': cross_details['price_change_since_cross'] if cross_details else None,
                        'current_time': current_row['timestamp'],
                        'ema_diff_pct': current_row['ema_diff_pct'],
                        'volume': current_row['volume']
                    }
                    
                    results.append(result)
                    print(f"✓ {normalized_symbol}: Valid signal found! (Cross {candles_since} candles ago)")
                
                # Rate limiting
                time.sleep(0.1)
                
            except Exception as e:
                print(f"Error analyzing {symbol}: {str(e)}")
                continue
        
        return results
    
    def display_results(self, results):
        """Display analysis results in a formatted way"""
        if not results:
            print("\nNo symbols found matching the criteria.")
            return []
        
        # Sort by candles since cross (most recent first)
        results.sort(key=lambda x: x['candles_since_cross'])
        
        print(f"\n{'='*80}")
        print(f"Found {len(results)} symbols with valid EMA cross signals:")
        print(f"{'='*80}\n")
        
        for r in results:
            print(f"Symbol: {r['symbol']}")
            print(f"  Current Price: ${r['current_price']:.8f}")
            print(f"  EMA9: ${r['ema9']:.8f} | EMA15: ${r['ema15']:.8f}")
            print(f"  Candles Since Cross: {r['candles_since_cross']}")
            print(f"  Cross Price: ${r['cross_price']:.8f}")
            print(f"  Price Change Since Cross: {r['price_change_pct']:.8f}%")
            print(f"  EMA Difference: {r['ema_diff_pct']:.8f}%")
            print(f"  Cross Time: {r['cross_time']}")
            print(f"  Volume: {r['volume']:.8f}")
            print("-" * 40)
        
        return results

def EMAs_9_15_Filter(
    symbols,
    timeframe='15m', 
    limit=200, 
    max_cross_candles=3, 
    max_price=None,
    show_details=True
):
    """
    Filter specific coins for EMA9 golden cross above EMA15 within specified candles
    with current price above both EMAs
    
    Parameters:
    symbols: list - List of symbols to analyze (e.g., ['BTC', 'ETH', 'ADA'])
    timeframe: str - Candlestick timeframe (default '15m')
    limit: int - Number of candles to fetch (default 200)
    max_cross_candles: int - Maximum candles since golden cross (default 3)
    max_price: float - Maximum price filter (default None for no filter)
    show_details: bool - Print detailed results (default True)
    
    Returns:
    pandas DataFrame - Filtered coins meeting ALL criteria
    """
    try:
        if not symbols:
            print("No symbols provided.")
            return pd.DataFrame()
        
        # Initialize analyzer
        analyzer = BinanceEMAAnalyzer()
        
        # Load markets
        print("Loading Binance markets...")
        analyzer.exchange.load_markets()
        
        # Analyze symbols for EMA conditions
        raw_results = analyzer.analyze_symbols(
            symbols=symbols,
            timeframe=timeframe,
            limit=limit,
            max_cross_candles=max_cross_candles,
            max_price=max_price
        )
        
        # Display results if requested
        if show_details:
            final_results = analyzer.display_results(raw_results)
        else:
            final_results = raw_results
        
        # Convert to DataFrame
        df_results = pd.DataFrame(final_results) if final_results else pd.DataFrame()
        
        if not df_results.empty:
            # Sort by candles since cross (most recent cross first)
            df_results = df_results.sort_values('candles_since_cross', ascending=True)
        
        return df_results
        
    except Exception as e:
        print(f"Error in EMAs_9_15_Filter: {str(e)}")
        return pd.DataFrame()

In [3]:
import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

class BinanceUptrendScanner:
    def __init__(self, timeframe='5m', exchange='binance'):
        """
        Initialize the uptrend scanner using CCXT
        
        Parameters:
        - timeframe: Candle timeframe for analysis ('1m', '5m', '15m', '30m', '1h', '4h', '1d')
        - exchange: Exchange name (default: 'binance')
        """
        self.exchange_name = exchange
        self.exchange = getattr(ccxt, exchange)({
            'enableRateLimit': True,
            'options': {
                'defaultType': 'spot',
            }
        })
        self.timeframe = timeframe
        
    def load_markets(self):
        """Load market data"""
        if not self.exchange.markets:
            self.exchange.load_markets()
    
    def get_historical_data(self, symbol, limit=100):
        """Fetch historical OHLCV data for a symbol"""
        try:
            # Fetch OHLCV data
            ohlcv = self.exchange.fetch_ohlcv(symbol, self.timeframe, limit=limit)
            
            # Convert to DataFrame
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.set_index('timestamp', inplace=True)
            
            return df
            
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
            return None
    
    def calculate_indicators(self, df):
        """Calculate technical indicators for trend detection"""
        if df is None or len(df) < 20:
            return None
            
        # Simple Moving Averages
        df['SMA_10'] = df['close'].rolling(window=10).mean()
        df['SMA_20'] = df['close'].rolling(window=20).mean()
        df['SMA_50'] = df['close'].rolling(window=50).mean() if len(df) >= 50 else np.nan
        
        # Exponential Moving Averages
        df['EMA_10'] = df['close'].ewm(span=10, adjust=False).mean()
        df['EMA_20'] = df['close'].ewm(span=20, adjust=False).mean()
        
        # RSI
        df['RSI'] = self.calculate_rsi(df['close'])
        
        # MACD
        df['MACD'], df['MACD_signal'], df['MACD_diff'] = self.calculate_macd(df['close'])
        
        # Volume indicators
        df['volume_sma'] = df['volume'].rolling(window=20).mean()
        df['volume_ratio'] = df['volume'] / df['volume_sma']
        
        # Bollinger Bands
        df['BB_middle'] = df['SMA_20']
        bb_std = df['close'].rolling(window=20).std()
        df['BB_upper'] = df['BB_middle'] + (bb_std * 2)
        df['BB_lower'] = df['BB_middle'] - (bb_std * 2)
        df['BB_width'] = df['BB_upper'] - df['BB_lower']
        
        return df
    
    def calculate_rsi(self, prices, period=14):
        """Calculate Relative Strength Index"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi
    
    def calculate_macd(self, prices, fast=12, slow=26, signal=9):
        """Calculate MACD indicator"""
        ema_fast = prices.ewm(span=fast, adjust=False).mean()
        ema_slow = prices.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        macd_diff = macd - macd_signal
        return macd, macd_signal, macd_diff
    
    def is_uptrending(self, df):
        """
        Determine if a coin is in an uptrend based on multiple criteria
        """
        if df is None or len(df) < 20:
            return False, {}
        
        latest = df.iloc[-1]
        
        # Calculate criteria
        criteria = {
            'price_above_sma20': latest['close'] > latest['SMA_20'],
            'sma10_above_sma20': latest['SMA_10'] > latest['SMA_20'],
            'ema_bullish': latest['EMA_10'] > latest['EMA_20'],
            'rsi_healthy': 40 < latest['RSI'] < 70,
            'macd_bullish': latest['MACD'] > latest['MACD_signal'],
            'volume_increasing': latest['volume_ratio'] > 1.2,
            'higher_lows': df['low'].iloc[-5:].min() > df['low'].iloc[-10:-5].min(),
            'above_bb_middle': latest['close'] > latest['BB_middle'],
            'momentum_strong': df['close'].iloc[-3:].mean() > df['close'].iloc[-6:-3].mean()
        }
        
        # Price momentum
        price_change_5 = (latest['close'] - df['close'].iloc[-5]) / df['close'].iloc[-5] * 100
        price_change_10 = (latest['close'] - df['close'].iloc[-10]) / df['close'].iloc[-10] * 100
        
        criteria['momentum_5_positive'] = price_change_5 > 0
        criteria['momentum_10_positive'] = price_change_10 > 0
        
        # Count how many criteria are met
        score = sum(criteria.values())
        
        # Consider it uptrending if at least 7 out of 11 criteria are met
        is_uptrend = score >= 7
        
        # Get 24h volume in USDT
        volume_24h = df['volume'].iloc[-24:].sum() if len(df) >= 24 else df['volume'].sum()
        volume_24h_usdt = volume_24h * latest['close']
        
        return is_uptrend, {
            'score': score,
            'price_change_5': round(price_change_5, 2),
            'price_change_10': round(price_change_10, 2),
            'current_price': latest['close'],
            'rsi': round(latest['RSI'], 2),
            'volume_ratio': round(latest['volume_ratio'], 2),
            'volume_24h_usdt': round(volume_24h_usdt, 2),
            'sma_20': round(latest['SMA_20'], 8),
            'ema_10': round(latest['EMA_10'], 8),
            'ema_20': round(latest['EMA_20'], 8),
            'macd': round(latest['MACD'], 8),
            'macd_signal': round(latest['MACD_signal'], 8),
            'bb_upper': round(latest['BB_upper'], 8),
            'bb_lower': round(latest['BB_lower'], 8),
            **{f'criteria_{k}': v for k, v in criteria.items()}
        }
    
    def scan_symbol(self, symbol):
        """Scan a single symbol for uptrend"""
        df = self.get_historical_data(symbol)
        if df is None:
            return None
            
        df = self.calculate_indicators(df)
        is_uptrend, analysis = self.is_uptrending(df)
        
        return {
            'symbol': symbol,
            'is_uptrending': is_uptrend,
            **analysis
        }
    
    def scan_symbols(self, symbols):
        """
        Scan a list of symbols and return DataFrame with all results
        
        Parameters:
        - symbols: List of trading pairs (e.g., ['BTC/USDT', 'ETH/USDT'] or ['BTCUSDT', 'ETHUSDT'])
        
        Returns:
        - DataFrame with all coins (uptrending and non-uptrending)
        """
        try:
            # Load markets if not loaded
            self.load_markets()
            
            # Convert symbol format if needed
            formatted_symbols = []
            for symbol in symbols:
                # Check if symbol needs formatting (no slash)
                if '/' not in symbol and symbol.endswith('USDT'):
                    # Convert BTCUSDT to BTC/USDT
                    base = symbol[:-4]  # Remove 'USDT'
                    formatted_symbol = f"{base}/USDT"
                    formatted_symbols.append(formatted_symbol)
                else:
                    formatted_symbols.append(symbol)
            
            print(f"Scanning {len(formatted_symbols)} symbols...")
            
            # Scan all symbols
            results = []
            for i, symbol in enumerate(formatted_symbols):
                if i % 10 == 0 and i > 0:
                    print(f"Progress: {i}/{len(formatted_symbols)}")
                
                result = self.scan_symbol(symbol)
                if result:
                    # Store original symbol format in result
                    result['original_symbol'] = symbols[i]
                    results.append(result)
                
                # Rate limiting
                if i % 3 == 0:
                    time.sleep(0.1)
            
            # Create DataFrame
            if results:
                df = pd.DataFrame(results)
                
                # Sort by score (descending)
                df = df.sort_values('score', ascending=False)
                
                # Add scan timestamp
                df['scan_time'] = datetime.now()
                
                # Reorder columns
                column_order = [
                    'symbol', 'original_symbol', 'is_uptrending', 'score', 'current_price', 'price_change_5', 
                    'price_change_10', 'rsi', 'volume_ratio', 'volume_24h_usdt', 'sma_20', 
                    'ema_10', 'ema_20', 'macd', 'macd_signal', 'bb_upper', 'bb_lower', 'scan_time'
                ]
                
                # Add criteria columns
                criteria_cols = [col for col in df.columns if col.startswith('criteria_')]
                column_order.extend(criteria_cols)
                
                # Reorder
                available_cols = [col for col in column_order if col in df.columns]
                df = df[available_cols]
                
                return df
            else:
                # Return empty DataFrame with correct structure
                return pd.DataFrame(columns=[
                    'symbol', 'is_uptrending', 'score', 'current_price', 'price_change_5', 
                    'price_change_10', 'rsi', 'volume_ratio', 'scan_time'
                ])
            
        except Exception as e:
            print(f"Error in scan: {e}")
            import traceback
            traceback.print_exc()
            return pd.DataFrame()


# Simple function to scan your coin list
def scan_coins(coin_list, timeframe='5m', exchange='binance', uptrend_only=True):
    """
    Scan a list of coins and return DataFrame
    
    Parameters:
    - coin_list: List of symbols (e.g., ['BTC/USDT', 'ETH/USDT', 'BNB/USDT'])
    - timeframe: Candle timeframe ('1m', '5m', '15m', '30m', '1h', '4h', '1d')
    - exchange: Exchange name (default: 'binance')
    - uptrend_only: If True, return only uptrending coins; if False, return all
    
    Returns:
    - DataFrame with coin analysis
    """
    scanner = BinanceUptrendScanner(timeframe=timeframe, exchange=exchange)
    df = scanner.scan_symbols(coin_list)
    
    if uptrend_only and not df.empty:
        # Filter only uptrending coins
        df = df[df['is_uptrending'] == True].copy()
    
    return df


# Helper function to create symbol list from base currencies
def create_symbol_list(base_currencies, quote_currency='USDT'):
    """
    Helper to create symbol list from base currencies
    
    Example:
    create_symbol_list(['BTC', 'ETH', 'BNB']) -> ['BTC/USDT', 'ETH/USDT', 'BNB/USDT']
    """
    return [f"{base}/{quote_currency}" for base in base_currencies]


# Helper function to convert Binance format to CCXT format
def convert_binance_symbols(binance_symbols):
    """
    Convert Binance format symbols to CCXT format
    
    Example:
    convert_binance_symbols(['BTCUSDT', 'ETHUSDT']) -> ['BTC/USDT', 'ETH/USDT']
    """
    ccxt_symbols = []
    for symbol in binance_symbols:
        if symbol.endswith('USDT'):
            base = symbol[:-4]
            ccxt_symbols.append(f"{base}/USDT")
        elif symbol.endswith('BTC'):
            base = symbol[:-3]
            ccxt_symbols.append(f"{base}/BTC")
        elif symbol.endswith('ETH'):
            base = symbol[:-3]
            ccxt_symbols.append(f"{base}/ETH")
        else:
            # If format is unknown, keep as is
            ccxt_symbols.append(symbol)
    return ccxt_symbols

In [4]:
import ccxt
import pandas as pd
import pandas_ta as ta
import time
import logging
from datetime import datetime
import os
from dotenv import load_dotenv
import threading

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()
API_KEY = os.getenv('BINANCE_API_KEY')
API_SECRET = os.getenv('BINANCE_API_SECRET')

class AutoTrader:
    def __init__(self):
        """Initialize auto trader"""
        # Initialize exchange
        self.exchange = ccxt.binance({
            'enableRateLimit': True,
            'options': {'defaultType': 'spot'},
            'apiKey': API_KEY,
            'secret': API_SECRET
        })
        
        # Trading parameters
        self.timeframe = '15m'
        self.risk_reward_ratio = 2.5
        
        # Track active positions
        self.positions = {}
        
        # Start monitoring
        self.monitoring = True
        self.monitor_thread = threading.Thread(target=self._monitor_loop)
        self.monitor_thread.daemon = True
        self.monitor_thread.start()
        
        logger.info("Auto Trader Ready!")
    
    def trade(self, coin_pair, amount):
        """
        Execute a trade with coin pair and dollar amount
        Example: trade('BTCUSDT', 100) or trade('ETHUSDT', 50)
        """
        try:
            # Convert BTCUSDT to BTC/USDT format
            coin_pair = coin_pair.upper()
            if coin_pair.endswith('USDT'):
                coin = coin_pair[:-4]  # Remove USDT
                symbol = f"{coin}/USDT"
            else:
                # If not in correct format, assume it's just the coin name
                symbol = f"{coin_pair}/USDT"
                coin = coin_pair
            
            # Check if already in position
            if symbol in self.positions:
                logger.warning(f"Already trading {symbol}")
                return
            
            # Check balance
            balance = self.exchange.fetch_balance()
            usdt_balance = balance['USDT']['free']
            
            if usdt_balance < amount:
                logger.error(f"Not enough balance. Have ${usdt_balance:.8f}, need ${amount}")
                return
            
            # Get current price and ATR
            ohlcv = self.exchange.fetch_ohlcv(symbol, self.timeframe, limit=50)
            df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
            df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
            
            current_price = df['close'].iloc[-1]
            atr_value = df['atr'].iloc[-1]
            
            # Calculate position size
            position_size = amount / current_price
            markets = self.exchange.load_markets()
            market = markets[symbol]
            
            # Handle precision - convert to int if needed
            precision = market['precision']['amount']
            if isinstance(precision, float):
                precision = int(-1 * float(f"{precision:e}".split('e')[1]))
            
            position_size = round(position_size, precision)
            
            # Calculate levels
            stop_distance = atr_value * 1.5
            stop_loss = current_price - stop_distance
            take_profit = current_price + (stop_distance * self.risk_reward_ratio)
            
            # Execute buy
            logger.info(f"\n{'='*50}")
            logger.info(f"🚀 BUYING {coin} with ${amount}")
            logger.info(f"{'='*50}")
            
            order = self.exchange.create_market_order(symbol, 'buy', position_size)
            entry_price = float(order['average']) if order['average'] else current_price
            
            # Recalculate with actual entry
            stop_loss = entry_price - stop_distance
            take_profit = entry_price + (stop_distance * self.risk_reward_ratio)
            
            # Store position
            self.positions[symbol] = {
                'coin': coin,
                'coin_pair': coin_pair,
                'entry_price': entry_price,
                'amount': position_size,
                'stop_loss': stop_loss,
                'take_profit': take_profit,
                'entry_time': datetime.now(),
                'stop_moved': False,
                'invested': amount
            }
            
            # Display info
            risk = stop_distance * position_size
            reward = (stop_distance * self.risk_reward_ratio) * position_size
            
            logger.info(f"✅ BOUGHT {position_size:.8f} {coin}")
            logger.info(f"💰 Entry Price: ${entry_price:.8f}")
            logger.info(f"🛑 Stop Loss: ${stop_loss:.8f} (Risk: ${risk:.8f})")
            logger.info(f"🎯 Take Profit: ${take_profit:.8f} (Reward: ${reward:.8f})")
            logger.info(f"📊 Risk/Reward: 1:{self.risk_reward_ratio}")
            logger.info(f"{'='*50}\n")
            
        except Exception as e:
            logger.error(f"Error: {e}")
    
    def status(self):
        """Show status of all positions"""
        if not self.positions:
            logger.info("No active trades")
            return
        
        logger.info(f"\n{'='*60}")
        logger.info("ACTIVE TRADES")
        logger.info(f"{'='*60}")
        
        for symbol, pos in self.positions.items():
            try:
                ticker = self.exchange.fetch_ticker(symbol)
                current = ticker['last']
                
                pnl = (current - pos['entry_price']) * pos['amount']
                pnl_pct = ((current - pos['entry_price']) / pos['entry_price']) * 100
                
                logger.info(f"\n{pos['coin']}:")
                logger.info(f"  Entry: ${pos['entry_price']:.8f} → Current: ${current:.8f}")
                logger.info(f"  P&L: ${pnl:.8f} ({pnl_pct:+.8f}%)")
                logger.info(f"  Stop: ${pos['stop_loss']:.8f} | Target: ${pos['take_profit']:.8f}")
                logger.info(f"  Stop at breakeven: {'Yes ✓' if pos['stop_moved'] else 'No'}")
                
            except Exception as e:
                logger.error(f"Error getting {symbol} status: {e}")
        
        logger.info(f"\n{'='*60}\n")
    
    def close(self, coin_pair):
        """Manually close a position - sells all available balance"""
        try:
            # Convert BTCUSDT to BTC/USDT format
            coin_pair = coin_pair.upper()
            if coin_pair.endswith('USDT'):
                coin = coin_pair[:-4]
                symbol = f"{coin}/USDT"
            else:
                symbol = f"{coin_pair}/USDT"
                coin = coin_pair
            
            # Get actual balance of the coin
            balance = self.exchange.fetch_balance()
            coin_balance = balance[coin]['free']
            
            if coin_balance <= 0:
                logger.info(f"No {coin} balance to sell")
                # Remove from positions if exists
                if symbol in self.positions:
                    del self.positions[symbol]
                return
            
            # Get market info for minimum order size
            markets = self.exchange.load_markets()
            market = markets[symbol]
            min_amount = market['limits']['amount']['min']
            
            if coin_balance < min_amount:
                logger.info(f"Balance too small: {coin_balance:.8f} {coin} (min: {min_amount})")
                if symbol in self.positions:
                    del self.positions[symbol]
                return
            
            # Get precision and round
            precision = market['precision']['amount']
            if isinstance(precision, float):
                precision = int(-1 * float(f"{precision:e}".split('e')[1]))
            
            sell_amount = round(coin_balance, precision)
            
            ticker = self.exchange.fetch_ticker(symbol)
            current = ticker['last']
            
            logger.info(f"Selling ALL {sell_amount:.6f} {coin}")
            
            # Sell all
            order = self.exchange.create_market_order(symbol, 'sell', sell_amount)
            exit_price = float(order['average']) if order['average'] else current
            
            # Calculate P&L if position exists
            if symbol in self.positions:
                pos = self.positions[symbol]
                pnl = (exit_price - pos['entry_price']) * sell_amount
                pnl_pct = ((exit_price - pos['entry_price']) / pos['entry_price']) * 100
                
                logger.info(f"\n{'='*50}")
                logger.info(f"CLOSED {coin} - SOLD ALL")
                logger.info(f"Amount Sold: {sell_amount:.6f} {coin}")
                logger.info(f"Exit Price: ${exit_price:.8f}")
                logger.info(f"P&L: ${pnl:.8f} ({pnl_pct:+.8f}%)")
                logger.info(f"{'='*50}\n")
                
                del self.positions[symbol]
            else:
                logger.info(f"\n{'='*50}")
                logger.info(f"SOLD ALL {coin}")
                logger.info(f"Amount: {sell_amount:.6f} {coin}")
                logger.info(f"Price: ${exit_price:.8f}")
                logger.info(f"Total: ${sell_amount * exit_price:.8f}")
                logger.info(f"{'='*50}\n")
            
        except Exception as e:
            logger.error(f"Error closing: {e}")
    
    def _monitor_loop(self):
        """Monitor positions automatically"""
        while self.monitoring:
            try:
                for symbol, pos in list(self.positions.items()):
                    ticker = self.exchange.fetch_ticker(symbol)
                    current_price = ticker['last']
                    
                    # Check if 15 minutes passed and move stop to breakeven
                    if not pos['stop_moved']:
                        time_passed = datetime.now() - pos['entry_time']
                        if time_passed.total_seconds() >= 900 and current_price > pos['entry_price']:
                            pos['stop_loss'] = pos['entry_price']
                            pos['stop_moved'] = True
                            logger.info(f"✅ {pos['coin']} stop moved to breakeven!")
                    
                    # Check stop loss
                    if current_price <= pos['stop_loss']:
                        logger.info(f"\n🛑 STOP LOSS HIT - {pos['coin']}")
                        logger.info(f"Exit Price: ${current_price:.8f}")
                        
                        # Get actual balance and sell all
                        balance = self.exchange.fetch_balance()
                        coin_balance = balance[pos['coin']]['free']
                        
                        if coin_balance > 0:
                            # Get market info
                            markets = self.exchange.load_markets()
                            market = markets[symbol]
                            min_amount = market['limits']['amount']['min']
                            
                            if coin_balance >= min_amount:
                                # Get precision
                                precision = market['precision']['amount']
                                if isinstance(precision, float):
                                    precision = int(-1 * float(f"{precision:e}".split('e')[1]))
                                sell_amount = round(coin_balance, precision)
                                
                                # Sell all
                                order = self.exchange.create_market_order(symbol, 'sell', sell_amount)
                                exit_price = float(order['average']) if order['average'] else current_price
                                actual_loss = (exit_price - pos['entry_price']) * sell_amount
                                logger.info(f"Sold {sell_amount:.6f} {pos['coin']} | Loss: ${actual_loss:.8f}")
                        
                        del self.positions[symbol]
                    
                    # Check take profit
                    elif current_price >= pos['take_profit']:
                        logger.info(f"\n🎯 TAKE PROFIT HIT - {pos['coin']}")
                        logger.info(f"Exit Price: ${current_price:.8f}")
                        
                        # Get actual balance and sell all
                        balance = self.exchange.fetch_balance()
                        coin_balance = balance[pos['coin']]['free']
                        
                        if coin_balance > 0:
                            # Get market info
                            markets = self.exchange.load_markets()
                            market = markets[symbol]
                            min_amount = market['limits']['amount']['min']
                            
                            if coin_balance >= min_amount:
                                # Get precision
                                precision = market['precision']['amount']
                                if isinstance(precision, float):
                                    precision = int(-1 * float(f"{precision:e}".split('e')[1]))
                                sell_amount = round(coin_balance, precision)
                                
                                # Sell all
                                order = self.exchange.create_market_order(symbol, 'sell', sell_amount)
                                exit_price = float(order['average']) if order['average'] else current_price
                                actual_profit = (exit_price - pos['entry_price']) * sell_amount
                                logger.info(f"Sold {sell_amount:.6f} {pos['coin']} | Profit: ${actual_profit:.8f}")
                        
                        del self.positions[symbol]
                
                time.sleep(30)  # Check every 30 seconds
                
            except Exception as e:
                logger.error(f"Monitor error: {e}")
                time.sleep(60)


# Create bot instance
bot = AutoTrader()

# Simple usage
print("\n🤖 AUTO TRADER READY!\n")
print("Commands:")
print("  bot.trade('BTCUSDT', 100)   # Buy $100 of Bitcoin")
print("  bot.trade('ETHUSDT', 50)    # Buy $50 of Ethereum")
print("  bot.status()                # Check all positions")
print("  bot.close('BTCUSDT')        # Manually close Bitcoin\n")

# The bot automatically:
# - Calculates position size and levels
# - Moves stop to breakeven after 15 minutes if in profit
# - Exits at stop loss or take profit
# - Monitors 24/7

2025-06-28 17:52:54,227 - INFO - Auto Trader Ready!



🤖 AUTO TRADER READY!

Commands:
  bot.trade('BTCUSDT', 100)   # Buy $100 of Bitcoin
  bot.trade('ETHUSDT', 50)    # Buy $50 of Ethereum
  bot.status()                # Check all positions
  bot.close('BTCUSDT')        # Manually close Bitcoin



In [5]:
from IPython.display import clear_output
import schedule
import time

def trade_and_track(symbol: str, amount: float, interval: int = 10):
    """
    Place a one-time trade, then clear+print bot.status() every `interval` seconds.
    
    Args:
    symbol: market symbol, e.g. 'OGUSDT'
    amount: quantity to buy
    interval: seconds between status updates
    """
    # 1) Execute your trade once
    bot.trade(symbol, amount)
    
    # 2) Define the status–display job
    def _show_status():
        clear_output(wait=True)
        bot.status()
    
    # 3) Clear any previously scheduled jobs, then schedule new one
    schedule.clear()
    schedule.every(interval).seconds.do(_show_status)
    
    # 4) Run one immediate status, then enter the scheduler loop
    _show_status()
    try:
        while True:
            schedule.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        print("🛑 Stopped tracking.")

In [6]:
df = pd.read_csv("Coin_List.csv", header=None)
coins = df[1:].iloc[:, 0].tolist()
print("Total Coins:", len(coins))

Total Coins: 291


In [66]:
uptrending_df = scan_coins(coins, timeframe='15m', uptrend_only=True)

Scanning 291 symbols...
Progress: 10/291
Progress: 20/291
Progress: 30/291
Progress: 40/291
Progress: 50/291
Progress: 60/291
Progress: 70/291
Progress: 80/291
Progress: 90/291
Progress: 100/291
Progress: 110/291
Progress: 120/291
Progress: 130/291
Progress: 140/291
Progress: 150/291
Progress: 160/291
Progress: 170/291
Progress: 180/291
Progress: 190/291
Progress: 200/291
Progress: 210/291
Progress: 220/291
Progress: 230/291
Progress: 240/291
Progress: 250/291
Progress: 260/291
Progress: 270/291
Progress: 280/291
Progress: 290/291


In [67]:
uptrending_df =  uptrending_df[(uptrending_df['score']>=8) & 
                                (uptrending_df['rsi'] <= 65) & 
                                (uptrending_df['criteria_macd_bullish'] == True) & 
                                (uptrending_df['volume_ratio'] >= 1) & 
                                (uptrending_df['criteria_higher_lows'] == True) 
                                ]

uptrending_df = uptrending_df[['original_symbol', 'score', 'rsi', 'criteria_macd_bullish',
                            'volume_ratio', 'criteria_higher_lows']]
uptrending_df

,original_symbol,score,rsi,criteria_macd_bullish,volume_ratio,criteria_higher_lows
201,SUSHIUSDT,11,62.50,True,1.87,True
146,YGGUSDT,10,62.50,True,1.29,True
207,THETAUSDT,10,58.33,True,1.11,True
12,BOMEUSDT,10,59.26,True,1.36,True
187,ARKMUSDT,9,64.29,True,1.02,True
270,NMRUSDT,9,58.33,True,1.55,True
25,XVGUSDT,9,64.15,True,1.26,True


In [68]:
results_df = EMAs_9_15_Filter(
    symbols=uptrending_df['original_symbol'].tolist(),
    timeframe='15m',
    limit=200,
    max_cross_candles=3,
    max_price=100,  # Only check coins under $100
    show_details=True
)

Loading Binance markets...
Analyzing 7 symbols...
✓ THETAUSDT: Valid signal found! (Cross 0 candles ago)
✓ NMRUSDT: Valid signal found! (Cross 0 candles ago)

Found 2 symbols with valid EMA cross signals:

Symbol: THETAUSDT
  Current Price: $0.66000000
  EMA9: $0.65724274 | EMA15: $0.65707267
  Candles Since Cross: 0
  Cross Price: $0.66000000
  Price Change Since Cross: 0.00000000%
  EMA Difference: 0.02588366%
  Cross Time: 2025-06-28 09:30:00
  Volume: 12827.10000000
----------------------------------------
Symbol: NMRUSDT
  Current Price: $7.12000000
  EMA9: $7.10372298 | EMA15: $7.10324424
  Candles Since Cross: 0
  Cross Price: $7.12000000
  Price Change Since Cross: 0.00000000%
  EMA Difference: 0.00673975%
  Cross Time: 2025-06-28 09:30:00
  Volume: 208.01000000
----------------------------------------


In [69]:
buy_pressure = get_binance_buy_pressure(
                    results_df['symbol'].tolist(),
                    top_n=10)

In [70]:
results_df = EMAs_9_15_Filter(
    symbols=buy_pressure['symbol'].tolist(),
    timeframe='15m',
    limit=200,
    max_cross_candles=3,
    max_price=100,  # Only check coins under $100
    show_details=True
)

Loading Binance markets...
Analyzing 2 symbols...
✓ THETAUSDT: Valid signal found! (Cross 0 candles ago)
✓ NMRUSDT: Valid signal found! (Cross 0 candles ago)

Found 2 symbols with valid EMA cross signals:

Symbol: THETAUSDT
  Current Price: $0.66000000
  EMA9: $0.65724274 | EMA15: $0.65707267
  Candles Since Cross: 0
  Cross Price: $0.66000000
  Price Change Since Cross: 0.00000000%
  EMA Difference: 0.02588366%
  Cross Time: 2025-06-28 09:30:00
  Volume: 12827.10000000
----------------------------------------
Symbol: NMRUSDT
  Current Price: $7.12000000
  EMA9: $7.10372298 | EMA15: $7.10324424
  Candles Since Cross: 0
  Cross Price: $7.12000000
  Price Change Since Cross: 0.00000000%
  EMA Difference: 0.00673975%
  Cross Time: 2025-06-28 09:30:00
  Volume: 208.01000000
----------------------------------------


In [71]:
results_df

,symbol,current_price,ema9,ema15,candles_since_cross,has_valid_signal,cross_time,cross_price,price_change_pct,current_time,ema_diff_pct,volume
0,THETAUSDT,0.66,0.657243,0.657073,0,True,2025-06-28 09:30:00,0.66,0.0,2025-06-28 09:30:00,0.025884,12827.10
1,NMRUSDT,7.12,7.103723,7.103244,0,True,2025-06-28 09:30:00,7.12,0.0,2025-06-28 09:30:00,0.006740,208.01


In [ ]:
trade_and_track('COSUSDT', 10)

2025-06-27 01:22:23,446 - INFO - 
2025-06-27 01:22:23,447 - INFO - ACTIVE TRADES
2025-06-27 01:22:23,447 - INFO - ============================================================
2025-06-27 01:22:23,536 - INFO - 
COS:
2025-06-27 01:22:23,536 - INFO -   Entry: $0.00 → Current: $0.00
2025-06-27 01:22:23,537 - INFO -   P&L: $0.04 (+0.4%)
2025-06-27 01:22:23,537 - INFO -   Stop: $0.00 | Target: $0.00
2025-06-27 01:22:23,537 - INFO -   Stop at breakeven: No
2025-06-27 01:22:23,538 - INFO - 

